# PG-LIF — One-shot memorization: confirmatory matched-control experiment (revision R1)
**Produces:** manuscript Section 7.4.

**Question.** The published 10-seed experiment showed that gating (vs. no gating) is decisive (+0.439, p = 0.001), but did not show that the *content* carried by the plateau gate matters: plateau vs. random-binary +0.020 (p = 0.17), plateau vs. permuted −0.000 (p = 0.17). Reviewer 3 asks for controls in which **sparsity, update frequency, and effective learning-rate scale are matched**, so that only the content-dependence of the gate differs.

**Primary and sensitivity runs.** The analysis below was fixed before seeds 100–149 were run, with the gate exactly as in the submitted experiment (`GATE_MODE = 'published'`); this run is **primary**. The submitted code computes that gate by dividing each neuron's storage-window mean of min(1, p) by the maximum of that quantity across neurons. That maximum is the gate's own theoretical maximum (0.8807, reached by every neuron whose dendrite fires at storage onset), and it took exactly this value for every pattern tested (2,000 patterns, at least 44 neurons tied), so the division is equivalent to a fixed constant and the gate is effectively neuron-local. The identical analysis was also run with the gate written as Eq. 12 (`GATE_MODE = 'eq12'`, normalised by p_max = P₀/(1 − α_p^{n_ref}) = 2.055) as a sensitivity check; its conclusions are the same.

**Revision history (stated for transparency).** The tests, seeds, rules and decision rules below were written before any of seeds 100–149 were run, and have not changed. The submitted-gate run was executed first. An intermediate draft then relabelled the Eq. 12 run as primary, based on a mistaken reading that the submitted gate was not neuron-local. Validation corrected that reading, and the label reverted to the original pre-specification. Only the "primary" label moved, and it moved back; no test, seed or decision rule was altered.

**Pre-registration (fixed before any of these seeds were run).**
- **Model, task, and every hyperparameter:** identical to the published experiment (κ = 0; N = 200; f = 0.10; 50%-masked cues; M ∈ {20, 50, 100, 200}).
- **Seeds:** 50 fresh seeds, 100–149, never used before. The published seeds 0–9 are *not* reused, so this is an independent confirmatory sample.
- **Fully paired design:** within a seed, all rules see the same patterns, the same dendritic weights, and the same recall cues; only the gate differs. (Control randomness uses its own generator, so it cannot shift the cue masks.)
- **Rules:**
  1. `plateau-graded` — the proposed gate.
  2. `mismatched-pattern` (**primary control**) — when storing pattern *m*, use the plateau gate that pattern *π(m)* produces, π a random derangement. This keeps the plateau gate's own sparsity, gradedness, per-pattern learning-rate scale, **and each neuron's total update count over the whole storage sequence exactly identical**; only the pattern→gate correspondence (the content) is broken.
  3. `permuted-graded` — same graded values on random neurons (published control; matches sparsity and learning-rate scale, but not per-neuron update frequency).
  4. `random-binary-lrmatched` — the same number of random neurons, each set to the mean non-zero plateau-gate value, so the summed gate (effective learning rate) is matched.
  5. `random-binary` — the published random-binary control (gate value 1), kept for continuity.
  6. `ungated` — replication of the headline comparison. Note: with every neuron gated identically, every row of W receives the same update, so this rule cannot discriminate patterns by construction (recall is exactly 1/M). It is a reference for symmetry breaking, not a test of interference.
- **Primary outcome:** per-seed recall pooled over M (as published).
- **Primary test (content-specificity):** plateau > mismatched-pattern, one-sided Wilcoxon signed-rank; family of three content controls (mismatched, permuted, random-binary-lrmatched) Holm-corrected at α = 0.05. Effect sizes are mean paired differences with 95% paired-bootstrap CIs (10,000 resamples).
- **Equivalence test:** TOST on the paired differences with margin δ = 0.02 recall (2 points), α = 0.05, so a null result is informative rather than merely non-significant.
- **Secondary (pre-specified):** does the plateau-vs-mismatched difference change with capacity? Per-seed slope of the difference against log₂M; two-sided Wilcoxon on the 50 slopes. (Motivated by the published per-M means; tested here only on fresh seeds.)
- **Decision rules for the manuscript wording:** (i) primary significant after Holm **and** CI excludes δ-negligible → claim a measurable content-specific advantage of the reported size; (ii) TOST establishes equivalence → state that the advantage is accounted for by gating structure and that content-specificity contributes less than 2 recall points; (iii) neither → report as inconclusive with the CI. The headline (gate vs. ungated) is reported regardless.

In [1]:
import os, json, time, math
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    BASE = os.path.join(ROOT, 'PG_LIF')
except Exception:
    BASE = './PG_LIF'
P3_ROOT = os.path.join(BASE, 'P3_results')
GATE_MODE = 'eq12'   # SENSITIVITY RUN (primary is 'published') (pre-specified); set to 'eq12' for the sensitivity run
OUT = os.path.join(P3_ROOT, 'one_shot_confirmatory_R1' + ('_eq12' if GATE_MODE == 'eq12' else ''))
os.makedirs(OUT, exist_ok=True)
print('Results folder:', OUT)

Results folder: ./PG_LIF/P3_results/one_shot_confirmatory_R1_eq12


## Model (copied unchanged from the published P3a notebook; device fixed to CPU)

In [2]:
import torch, numpy as np, matplotlib.pyplot as plt
def decay(tau): return math.exp(-1.0 / tau)
N_IN, N_MEM = 200, 200
F_ACT = 0.10
T_STORE, T_REC = 30, 30
TH_D = 1.0; TREF_P = 10
AP, AD, AM, AA = decay(15), decay(10), decay(20), decay(100)
ETA, WMAX = 0.6, 1.0
SEEDS = list(range(10))
DEV = 'cpu'   # R1: fixed to CPU so every seed reproduces bit-exactly on any machine
CFG = dict(N_IN=N_IN, N_MEM=N_MEM, F_ACT=F_ACT, T_STORE=T_STORE, T_REC=T_REC,
           TH_D=TH_D, TREF_P=TREF_P, tau_p=15, tau_d=10, tau_m=20, tau_a=100,
           ETA=ETA, WMAX=WMAX, seeds=SEEDS)


def make_patterns(Mn, dev):
    P = torch.zeros(Mn, N_IN, device=dev)
    for m in range(Mn): P[m, torch.randperm(N_IN, device=dev)[:int(F_ACT * N_IN)]] = 1.0
    return P

def run_layer(x_const, W, Wd, T, kappa=0.0):
    """PG-LIF memory layer. Default kappa=0.0 preserves the ORIGINAL experiment exactly (plateau computed
    for gating only, never touching the soma). kappa>0 activates the plateau's OTHER role -- its own
    contribution to the somatic membrane, exactly as in the main PGLIFCell (kappa*p_clamped*(1-alpha_m))
    -- so that BOTH of the plateau's roles (memory drive AND plasticity gate) are active simultaneously,
    which the original kappa=0 design never tested. Batch over rows of x_const."""
    B = x_const.shape[0]; dev = x_const.device
    z = lambda: torch.zeros(B, N_MEM, device=dev)
    vs, vd, p, a, rp, spk, gsum = z(), z(), z(), z(), z(), z(), z()
    P_MAX = 1.0 / (1 - AP ** TREF_P)        # Eq. 12 / Proposition 2 bound, P0 = 1
    for t in range(T):
        iff = x_const @ W.T; idd = x_const @ Wd.T
        vd = AD * vd + idd
        ed = ((vd >= TH_D) & (rp == 0)).float()
        rp = torch.clamp(rp - 1, min=0) + ed * TREF_P
        p = AP * p + ed
        p_clamped = torch.clamp(p, max=1.0)
        vs = AM * vs + iff + kappa * p_clamped * (1 - AM)
        th = 1.0 + a
        s = (vs >= th).float(); vs = vs - s * th; a = AA * a + s
        spk += s; gsum += (torch.clamp(p / P_MAX, max=1.0) if GATE_MODE == 'eq12' else p_clamped)
    return spk, gsum / T

def experiment(Mn, rule, mask_frac=0.5, binary_w=False, seed=0, kappa=0.0):
    torch.manual_seed(seed); np.random.seed(seed)
    pats = make_patterns(Mn, DEV)
    Wd = (torch.rand(N_MEM, N_IN, device=DEV) < 0.10).float() * 0.35
    W = torch.zeros(N_MEM, N_IN, device=DEV)
    for m in range(Mn):
        x = pats[m:m+1]
        _, g = run_layer(x, W, Wd, T_STORE, kappa=kappa); g = g[0]
        graded = (g > 0).float() * torch.clamp(g / g.max().clamp(min=1e-6), max=1.0)
        if rule == 'plateau-graded':
            gate = graded
        elif rule == 'permuted-graded':
            gate = graded[torch.randperm(N_MEM, device=DEV)]
        elif rule == 'random-binary':
            k = int((g > 0).float().sum().item())
            gate = torch.zeros(N_MEM, device=DEV); gate[torch.randperm(N_MEM, device=DEV)[:max(k, 1)]] = 1.0
        else:  # ungated
            gate = torch.ones(N_MEM, device=DEV)
        xb = x[0]; qp = xb; qm = 0.4 * (1 - xb)
        W = torch.clamp(W + ETA * gate.unsqueeze(1) * ((WMAX - W) * qp.unsqueeze(0) - W * qm.unsqueeze(0)), 0, WMAX)
        if binary_w: W = (W > 0.5).float()
    sigs = run_layer(pats, W, Wd, T_REC, kappa=kappa)[0]          # (M, N_MEM) full-cue signatures
    cues = pats.clone()
    for m in range(Mn):
        act = torch.nonzero(cues[m]).flatten()
        cues[m, act[torch.randperm(len(act), device=DEV)[:int(mask_frac * len(act))]]] = 0.0
    resp = run_layer(cues, W, Wd, T_REC, kappa=kappa)[0]
    sims = torch.nn.functional.cosine_similarity(resp.unsqueeze(1), sigs.unsqueeze(0), dim=2)
    return float((sims.argmax(1) == torch.arange(Mn, device=DEV)).float().mean())

RULES = ['plateau-graded', 'permuted-graded', 'random-binary', 'ungated']
def sweep(Ms, mask_frac=0.5, binary_w=False, rules=RULES, seeds=SEEDS):
    R = {r: {M_: [experiment(M_, r, mask_frac, binary_w, s) for s in seeds] for M_ in Ms} for r in rules}
    return R

## Matched-control storage and recall

In [3]:
RULES_R1 = ['plateau-graded', 'mismatched-pattern', 'permuted-graded',
            'random-binary-lrmatched', 'random-binary', 'ungated']
SEEDS_R1 = list(range(100, 150))
MS = [20, 50, 100, 200]
CFG_R1 = dict(CFG, seeds=SEEDS_R1, rules=RULES_R1, Ms=MS, mask_frac=0.5, device=DEV, gate_mode=GATE_MODE)
json.dump(CFG_R1, open(os.path.join(OUT, 'config_R1.json'), 'w'), indent=2)

def graded_gate(g):
    return (g > 0).float() * torch.clamp(g / g.max().clamp(min=1e-6), max=1.0)

def eq12_gate(g):
    # run_layer already returns the storage-window mean of min(1, p/p_max) when GATE_MODE == 'eq12'
    return g

def derangement(n, gen):
    while True:
        p = torch.randperm(n, generator=gen)
        if n == 1 or not (p == torch.arange(n)).any(): return p

def experiment_R1(Mn, seed, mask_frac=0.5):
    """All rules for one (M, seed), fully paired: same patterns, dendritic weights and cues for every rule."""
    torch.manual_seed(seed); np.random.seed(seed)
    pats = make_patterns(Mn, DEV)
    Wd = (torch.rand(N_MEM, N_IN, device=DEV) < 0.10).float() * 0.35
    cues = pats.clone()                                       # one cue set, shared by all rules
    for m in range(Mn):
        act = torch.nonzero(cues[m]).flatten()
        cues[m, act[torch.randperm(len(act), device=DEV)[:int(mask_frac * len(act))]]] = 0.0
    # kappa = 0: the plateau (hence the gate) depends only on the dendritic pathway, never on W
    G = [(eq12_gate if GATE_MODE == 'eq12' else graded_gate)(
             run_layer(pats[m:m+1], torch.zeros(N_MEM, N_IN, device=DEV), Wd, T_STORE)[1][0])
         for m in range(Mn)]
    out = {}
    for rule in RULES_R1:
        gen = torch.Generator().manual_seed(1_000_003 * seed + 7919 * Mn + RULES_R1.index(rule))
        perm_pat = derangement(Mn, gen) if rule == 'mismatched-pattern' else None
        W = torch.zeros(N_MEM, N_IN, device=DEV)
        n_active, g_sum, per_neuron = [], [], torch.zeros(N_MEM)
        for m in range(Mn):
            x = pats[m:m+1]; gr = G[m]; k = int((gr > 0).sum())
            if rule == 'plateau-graded':
                gate = gr
            elif rule == 'mismatched-pattern':
                gate = G[int(perm_pat[m])]
            elif rule == 'permuted-graded':
                gate = gr[torch.randperm(N_MEM, generator=gen)]
            elif rule == 'random-binary-lrmatched':
                gate = torch.zeros(N_MEM)
                if k > 0: gate[torch.randperm(N_MEM, generator=gen)[:k]] = float(gr[gr > 0].mean())
            elif rule == 'random-binary':                      # published control, unchanged (incl. max(k, 1))
                gate = torch.zeros(N_MEM); gate[torch.randperm(N_MEM, generator=gen)[:max(k, 1)]] = 1.0
            else:
                gate = torch.ones(N_MEM)
            n_active.append(int((gate > 0).sum())); g_sum.append(float(gate.sum())); per_neuron += gate
            xb = x[0]; qp = xb; qm = 0.4 * (1 - xb)
            W = torch.clamp(W + ETA * gate.unsqueeze(1) * ((WMAX - W) * qp.unsqueeze(0) - W * qm.unsqueeze(0)), 0, WMAX)
        sigs = run_layer(pats, W, Wd, T_REC)[0]
        resp = run_layer(cues, W, Wd, T_REC)[0]
        sims = torch.nn.functional.cosine_similarity(resp.unsqueeze(1), sigs.unsqueeze(0), dim=2)
        out[rule] = {'recall': float((sims.argmax(1) == torch.arange(Mn)).float().mean()),
                     'mean_active': float(np.mean(n_active)), 'mean_gate_sum': float(np.mean(g_sum)),
                     'per_neuron_total_cv': float(per_neuron.std() / per_neuron.mean().clamp(min=1e-9)),
                     'per_neuron_total': per_neuron.tolist()}
    return out

## Run (resumable per seed)

In [4]:
t0 = time.time()
RAW = {}
for sd in SEEDS_R1:
    f = os.path.join(OUT, f'seed{sd}.json')
    if os.path.exists(f):
        RAW[sd] = json.load(open(f)); continue
    RAW[sd] = {str(M_): experiment_R1(M_, sd) for M_ in MS}
    json.dump(RAW[sd], open(f, 'w'))
    if (sd - SEEDS_R1[0]) % 10 == 9: print(f'seed {sd} done ({time.time()-t0:.0f}s)')
print(f'all {len(SEEDS_R1)} seeds done ({time.time()-t0:.0f}s)')

seed 109 done (18s)


seed 119 done (37s)


seed 129 done (55s)


seed 139 done (74s)


seed 149 done (93s)
all 50 seeds done (93s)


## Matching diagnostics (does each control match what it claims to match?)

In [5]:
print(f"{'rule':26s}{'active/pattern':>16s}{'gate sum/pattern':>18s}{'per-neuron CV':>15s}{'max |per-neuron diff| vs plateau':>34s}")
MATCH = {}
for r in RULES_R1:
    act = np.mean([RAW[s][str(M_)][r]['mean_active'] for s in SEEDS_R1 for M_ in MS])
    gs = np.mean([RAW[s][str(M_)][r]['mean_gate_sum'] for s in SEEDS_R1 for M_ in MS])
    cv = np.mean([RAW[s][str(M_)][r]['per_neuron_total_cv'] for s in SEEDS_R1 for M_ in MS])
    dmax = max(float(np.max(np.abs(np.array(RAW[s][str(M_)][r]['per_neuron_total']) -
                                   np.array(RAW[s][str(M_)]['plateau-graded']['per_neuron_total']))))
               for s in SEEDS_R1 for M_ in MS)
    MATCH[r] = dict(mean_active=act, mean_gate_sum=gs, per_neuron_cv=cv, max_per_neuron_abs_diff=dmax)
    print(f'{r:26s}{act:16.2f}{gs:18.3f}{cv:15.3f}{dmax:34.4f}')
json.dump(MATCH, open(os.path.join(OUT, 'matching_R1.json'), 'w'), indent=2)

rule                        active/pattern  gate sum/pattern  per-neuron CV  max |per-neuron diff| vs plateau
plateau-graded                      175.77            84.233          0.095                            0.0000
mismatched-pattern                  175.77            84.233          0.095                            0.0001
permuted-graded                     175.77            84.233          0.050                           45.5749
random-binary-lrmatched             175.77            84.233          0.049                           40.4842
random-binary                       175.77           175.773          0.050                          138.0681
ungated                             200.00           200.000          0.000                          158.0681


## Pre-specified statistics

In [6]:
from scipy import stats
rng_boot = np.random.default_rng(12345)
DELTA = 0.02
pooled = {r: np.array([np.mean([RAW[s][str(M_)][r]['recall'] for M_ in MS]) for s in SEEDS_R1]) for r in RULES_R1}
def boot_ci(d, n=10000):
    idx = rng_boot.integers(0, len(d), size=(n, len(d)))
    b = d[idx].mean(1); return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))
def tost(d, delta=DELTA):
    n = len(d); m = d.mean(); se = d.std(ddof=1) / np.sqrt(n)
    p_low = 1 - stats.t.cdf((m + delta) / se, n - 1)       # H0: mean <= -delta
    p_up = stats.t.cdf((m - delta) / se, n - 1)            # H0: mean >= +delta
    return float(max(p_low, p_up))
def sign_one_sided(d):
    n = int((d != 0).sum()); k = int((d > 0).sum())
    return float(stats.binomtest(k, n, 0.5, alternative='greater').pvalue) if n else 1.0
RESULTS = {}
fam = ['mismatched-pattern', 'permuted-graded', 'random-binary-lrmatched']
for c in fam + ['random-binary', 'ungated']:
    d = pooled['plateau-graded'] - pooled[c]
    w = stats.wilcoxon(d, alternative='greater', zero_method='wilcox') if np.any(d != 0) else None
    sd = d.std(ddof=1)
    RESULTS[c] = dict(mean_diff=float(d.mean()), ci95=boot_ci(d), sd_diff=float(sd),
                      p_wilcoxon_one_sided=float(w.pvalue) if w else 1.0, p_sign_one_sided=sign_one_sided(d),
                      p_tost_delta_0p02=tost(d) if sd > 0 else 0.0,
                      mde_80pct_power=float((stats.norm.ppf(0.95) + stats.norm.ppf(0.8)) * sd / np.sqrt(len(d))),
                      n_positive=int((d > 0).sum()), n_negative=int((d < 0).sum()), n=len(d))
# Holm across the three content controls
ps = sorted([(RESULTS[c]['p_wilcoxon_one_sided'], c) for c in fam])
running = 0.0
for i, (p, c) in enumerate(ps):
    running = max(running, min(1.0, (len(ps) - i) * p)); RESULTS[c]['p_holm'] = running
# Secondary: capacity dependence of plateau - mismatched
logM = np.log2(MS)
slopes = np.array([np.polyfit(logM, [RAW[s][str(M_)]['plateau-graded']['recall'] -
                                     RAW[s][str(M_)]['mismatched-pattern']['recall'] for M_ in MS], 1)[0]
                   for s in SEEDS_R1])
ws = stats.wilcoxon(slopes, alternative='two-sided') if np.any(slopes != 0) else None
RESULTS['secondary_slope_vs_log2M'] = dict(mean_slope=float(slopes.mean()), ci95=boot_ci(slopes),
                                          p_wilcoxon_two_sided=float(ws.pvalue) if ws else 1.0)
perM = {}
for M_ in MS:
    perM[M_] = {r: (float(np.mean([RAW[s][str(M_)][r]['recall'] for s in SEEDS_R1])),
                    float(np.std([RAW[s][str(M_)][r]['recall'] for s in SEEDS_R1])))
                for r in RULES_R1}
RESULTS['per_M_mean_std'] = perM
json.dump(RESULTS, open(os.path.join(OUT, 'stats_R1.json'), 'w'), indent=2)

print('Per-M recall, mean ± std over 50 seeds')
print(f"{'M':>5}" + ''.join(f'{r:>26s}' for r in RULES_R1))
for M_ in MS:
    print(f'{M_:>5}' + ''.join(f'{perM[M_][r][0]:>19.3f} ± {perM[M_][r][1]:.3f}' for r in RULES_R1))
print('\nplateau-graded minus control (pooled over M; 50 fresh seeds)')
for c in fam + ['random-binary', 'ungated']:
    R = RESULTS[c]
    print(f"{c:26s} diff {R['mean_diff']:+.4f} [{R['ci95'][0]:+.4f}, {R['ci95'][1]:+.4f}]  "
          f"Wilcoxon p={R['p_wilcoxon_one_sided']:.4g}" + (f"  Holm p={R['p_holm']:.4g}" if 'p_holm' in R else '') +
          f"  TOST(±0.02) p={R['p_tost_delta_0p02']:.4g}  +/-: {R['n_positive']}/{R['n_negative']}  MDE80={R['mde_80pct_power']:.4f}")
S = RESULTS['secondary_slope_vs_log2M']
print(f"\nSecondary: slope of (plateau - mismatched) per doubling of M = {S['mean_slope']:+.4f} "
      f"[{S['ci95'][0]:+.4f}, {S['ci95'][1]:+.4f}], two-sided Wilcoxon p = {S['p_wilcoxon_two_sided']:.4g}")

Per-M recall, mean ± std over 50 seeds
    M            plateau-graded        mismatched-pattern           permuted-graded   random-binary-lrmatched             random-binary                   ungated
   20              0.997 ± 0.012              0.994 ± 0.016              0.995 ± 0.015              0.995 ± 0.015              0.782 ± 0.094              0.050 ± 0.000
   50              0.764 ± 0.048              0.747 ± 0.061              0.723 ± 0.050              0.706 ± 0.053              0.442 ± 0.055              0.020 ± 0.000
  100              0.525 ± 0.047              0.529 ± 0.050              0.504 ± 0.048              0.495 ± 0.048              0.277 ± 0.042              0.010 ± 0.000
  200              0.356 ± 0.041              0.364 ± 0.039              0.348 ± 0.036              0.334 ± 0.039              0.183 ± 0.027              0.005 ± 0.000

plateau-graded minus control (pooled over M; 50 fresh seeds)
mismatched-pattern         diff +0.0020 [-0.0031, +0.0070]  Wilco

## Figure (300 dpi)

In [7]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
labels = ['mismatched-pattern', 'permuted-graded', 'random-binary-lrmatched', 'random-binary']
for i, c in enumerate(labels):
    R = RESULTS[c]
    ax[0].errorbar(R['mean_diff'], i, xerr=[[R['mean_diff'] - R['ci95'][0]], [R['ci95'][1] - R['mean_diff']]],
                   fmt='o', color='k', capsize=4)
ax[0].axvline(0, color='grey', lw=0.8); ax[0].axvspan(-DELTA, DELTA, color='grey', alpha=0.12, label='±0.02 equivalence margin')
ax[0].set_yticks(range(len(labels))); ax[0].set_yticklabels(labels); ax[0].invert_yaxis()
ax[0].set_xlabel('recall: plateau-graded minus control (pooled over M), 95% CI'); ax[0].legend(fontsize=8, loc='upper right')
for c in ['mismatched-pattern', 'permuted-graded', 'random-binary-lrmatched']:
    dm = [np.mean([RAW[s][str(M_)]['plateau-graded']['recall'] - RAW[s][str(M_)][c]['recall'] for s in SEEDS_R1]) for M_ in MS]
    se = [np.std([RAW[s][str(M_)]['plateau-graded']['recall'] - RAW[s][str(M_)][c]['recall'] for s in SEEDS_R1], ddof=1) / np.sqrt(len(SEEDS_R1)) for M_ in MS]
    ax[1].errorbar(MS, dm, yerr=1.96 * np.array(se), marker='o', ms=4, capsize=3, label=f'vs {c}')
ax[1].axhline(0, color='grey', lw=0.8); ax[1].set_xscale('log', base=2); ax[1].set_xticks(MS); ax[1].set_xticklabels(MS)
ax[1].set_xlabel('stored patterns M'); ax[1].set_ylabel('recall difference (mean ± 95% CI)'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_oneshot_matched_controls.png'), dpi=300); plt.show()

### Interpretation
Apply the pre-specified decision rules in the first cell to `stats_R1.json`; do not substitute a different test after seeing the data. The matching-diagnostics table must show that `mismatched-pattern` has a per-neuron total identical to `plateau-graded` (max difference 0) and the same gate sum per pattern; otherwise the control is not doing its job and the comparison is not reported as matched.